In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip -q "/content/drive/My Drive/data-project-s/Test_State.zip" -d /content/extracted_data

In [ ]:
!pip install pyspark

import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time
import pandas
import csv

In [ ]:

# =====================================================================
# 3. Running Spark
# =====================================================================
print("⚡ Running Appachi of Spark...")
spark = SparkSession.builder \
    .appName("mobility-platform-colab") \
    .config("spark.driver.memory", "4g") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# =====================================================================
# 4. A loop for running 35 Round
# =====================================================================
base_path = "/content/extracted_data/Test_State"
spark_results = []

print("\n🚀 Start processing 36 Rounds with for structure...")

for i in range(1, 37):
    state_dir = os.path.join(base_path, f"Test_{i}")

    u_file = os.path.join(state_dir, f"01_users_state_{i}.csv")
    t_file = os.path.join(state_dir, f"02_trips_state_{i}.csv")

#If there is not any folder or is one problem go to another round
    if not os.path.exists(u_file) or not os.path.exists(t_file):
        print(f"  {i} ")
        continue

    try:
        # reading files for each round
        df_users = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(u_file)
        df_trips = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(t_file)

        # Starting the exact ime for timer
        start_time = time.time()

        # Calculating the exact time for query
        trip_duration_seconds = df_trips["end_time"].cast("long") - df_trips["start_time"].cast("long")
        df_trips_with_duration = df_trips.withColumn("duration", trip_duration_seconds)

        # running query 2
        query2_stats = df_users.join(
            df_trips_with_duration,
            df_users["user_id"] == df_trips_with_duration["user_id"],
            "left"
        ).groupBy(
            df_users["user_id"], "name_user", "surname_user"
        ).agg(
            F.count(df_trips_with_duration["start_time"]).alias("total_trips"),
            F.avg(df_trips_with_duration["duration"]).alias("avg_tripduration")
        )

        # Force spark to execute for timer (Action)
        query2_stats.count()

        # End of timer
        elapsed_time = time.time() - start_time

        spark_results.append([i, elapsed_time])
        print(f" Round {i} is done with successfully in{elapsed_time:.4f} second")

    except Exception as e:
        print(f"❌ Round {i} is faced with the error: {e}")
        continue

# =====================================================================
# 5. Saving the final Excel files
# =====================================================================
output_perf_file = "/content/spark_query2_performance.csv"
header = ["Round", "Spark_Query2_Execution_Time"]

with open(output_perf_file, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(spark_results)

print(f"\n finish! file is saved successfully: {output_perf_file}")
print(" Can save/download file spark_query2_performance.csv.")

spark.stop()

⚡ در حال راه‌اندازی موتور آپاچی اسپارک...

🚀 شروع فرآیند پردازش خودکار روی ۳۶ راند...
✅ راند 1 با موفقیت در 2.5121 ثانیه پردازش شد.
✅ راند 2 با موفقیت در 0.8257 ثانیه پردازش شد.
✅ راند 3 با موفقیت در 0.5363 ثانیه پردازش شد.
✅ راند 4 با موفقیت در 0.3836 ثانیه پردازش شد.
✅ راند 5 با موفقیت در 0.3483 ثانیه پردازش شد.
✅ راند 6 با موفقیت در 0.3310 ثانیه پردازش شد.
✅ راند 7 با موفقیت در 0.4452 ثانیه پردازش شد.
✅ راند 8 با موفقیت در 0.5537 ثانیه پردازش شد.
✅ راند 9 با موفقیت در 0.4104 ثانیه پردازش شد.
✅ راند 10 با موفقیت در 0.2946 ثانیه پردازش شد.
✅ راند 11 با موفقیت در 0.3437 ثانیه پردازش شد.
✅ راند 12 با موفقیت در 0.2745 ثانیه پردازش شد.
✅ راند 13 با موفقیت در 0.3294 ثانیه پردازش شد.
✅ راند 14 با موفقیت در 0.2990 ثانیه پردازش شد.
✅ راند 15 با موفقیت در 0.3071 ثانیه پردازش شد.
✅ راند 16 با موفقیت در 0.3198 ثانیه پردازش شد.
✅ راند 17 با موفقیت در 0.3299 ثانیه پردازش شد.
✅ راند 18 با موفقیت در 0.2677 ثانیه پردازش شد.
✅ راند 19 با موفقیت در 0.2445 ثانیه پردازش شد.
✅ راند 20 با موفقیت در 0.2915 